In [3]:
from google.colab import files

uploaded = files.upload()

Saving questions_with_plans (2).json to questions_with_plans (2).json


In [4]:
# ════════════════════════════════════════════════════════════
#  CODE 1: Text → Plan  |  Evaluate Plan Token F1
#  Upload: lora_adapter.zip (planner), questions_with_plans_(2).json
# ════════════════════════════════════════════════════════════

!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3

import json, zipfile, os, re, gc, time
from collections import Counter
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from peft import PeftModel

# ── Config ────────────────────────────────────────────────────
ADAPTER_ZIP  = "./lora_adapter.zip"                    # planner weights
ADAPTER_DIR  = "./lora_adapter_planner"
DATA_PATH    = "./questions_with_plans (2).json"
OUTPUT_PATH  = "./code1_plan_results.json"
MAX_INPUT    = 512
MAX_TARGET   = 384
N_SAMPLES    = 101

# ── Unzip ─────────────────────────────────────────────────────
if not os.path.exists(ADAPTER_DIR):
    with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
        z.extractall(ADAPTER_DIR)
    print("Extracted to:", ADAPTER_DIR)

# ── Load model ────────────────────────────────────────────────
MODEL_NAME = "google/flan-t5-xl"
dtype = torch.bfloat16 if torch.cuda.is_available() and \
        torch.cuda.get_device_capability()[0] >= 8 else torch.float16

print(f"Loading base model ({dtype})...")
base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, device_map="auto",
)
print("Loading planner LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
model.config.use_cache = True
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print("Model ready!")

# ── Load data ─────────────────────────────────────────────────
with open(DATA_PATH, encoding="utf-8") as f:
    data = json.load(f)
samples = data[:N_SAMPLES]
print(f"Loaded {len(samples)} samples")

# ── Generate plans ────────────────────────────────────────────
def generate_plan(input_text):
    inp = tokenizer(input_text, return_tensors="pt",
                    max_length=MAX_INPUT, truncation=True)
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=MAX_TARGET,
                             num_beams=4, early_stopping=True, length_penalty=1.0)
    return tokenizer.decode(out[0], skip_special_tokens=True)

results = []
start = time.time()
for i, sample in enumerate(samples):
    pred_plan = generate_plan(sample["input"])
    results.append({
        "idx"       : i,
        "input"     : sample["input"],
        "gold_plan" : sample["output"],
        "pred_plan" : pred_plan,
        "gold_sql"  : sample["sql"],
        "db_id"     : sample.get("db_id", ""),
    })
    if (i + 1) % 25 == 0:
        elapsed = time.time() - start
        eta = elapsed / (i + 1) * (N_SAMPLES - i - 1)
        print(f"  [{i+1}/{N_SAMPLES}]  elapsed: {elapsed/60:.1f} min  ETA: {eta/60:.1f} min")

# ── Token F1 ──────────────────────────────────────────────────
def tokenize(text):
    return re.findall(r'\w+', text.lower())

def token_f1(pred, gold):
    pt = tokenize(pred); gt = tokenize(gold)
    if len(pt) == 0 and len(gt) == 0: return 1.0
    if len(pt) == 0 or  len(gt) == 0: return 0.0
    overlap   = sum((Counter(pt) & Counter(gt)).values())
    precision = overlap / len(pt)
    recall    = overlap / len(gt)
    if precision + recall == 0: return 0.0
    return 2 * precision * recall / (precision + recall)

f1_scores = []
for r in results:
    f1 = token_f1(r["pred_plan"], r["gold_plan"])
    f1_scores.append(f1)
    r["plan_f1"] = round(f1, 4)

avg_f1 = sum(f1_scores) / len(f1_scores)
print("\n══════════════════════════════════════")
print(f"   CODE 1: PLAN EVAL (n={len(results)})")
print("══════════════════════════════════════")
print(f"  Plan Token F1 : {avg_f1*100:.2f}%")
print("══════════════════════════════════════")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"Saved → {OUTPUT_PATH}")

print("\n── Samples ──")
for i in [0, 10, 50]:
    r = results[i]
    print(f"\n[{i}] Plan F1={r['plan_f1']:.3f}")
    print(f"  GOLD: {r['gold_plan']}")
    print(f"  PRED: {r['pred_plan']}")

Loading base model (torch.float16)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading planner LoRA...
Model ready!
Loaded 101 samples
  [25/101]  elapsed: 2.1 min  ETA: 6.4 min
  [50/101]  elapsed: 6.1 min  ETA: 6.2 min
  [75/101]  elapsed: 8.5 min  ETA: 2.9 min
  [100/101]  elapsed: 13.0 min  ETA: 0.1 min

══════════════════════════════════════
   CODE 1: PLAN EVAL (n=101)
══════════════════════════════════════
  Plan Token F1 : 90.74%
══════════════════════════════════════
Saved → ./code1_plan_results.json

── Samples ──

[0] Plan F1=1.000
  GOLD: step1: SCAN | table: product || step2: AGGREGATE | Scalar aggregate (no GROUP BY)  ->  compute COUNT(*) || step3: PROJECT | columns: COUNT(*)
  PRED: step1: SCAN | table: product || step2: AGGREGATE | Scalar aggregate (no GROUP BY) -> compute COUNT(*) || step3: PROJECT | columns: COUNT(*)

[10] Plan F1=1.000
  GOLD: step1: SCAN | table: orders || step2: AGGREGATE | Scalar aggregate (no GROUP BY)  ->  compute SUM(Total_Amount) || step3: PROJECT | columns: SUM(Total_Amount)
  PRED: step1: SCAN | table: orders || step2:

In [5]:
from google.colab import files
files.download('/content/code1_plan_results.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>